# Qwen2.5-Omni-7B Thinker - LoRA Fine-tuning

Korean speech audio input -> Character text response (Debi & Marlene)

**Pipeline:**
1. Discord user speaks Korean -> audio captured
2. Qwen2.5-Omni Thinker processes audio -> generates character text response
3. Text response -> CosyVoice3 TTS (separate system)

**Two options provided:**
- **Option A: LLaMA-Factory** (primary, PR #7537 merged)
- **Option B: ms-swift** (backup, official ModelScope support)

**Requirements:** Colab A100 (40GB+ VRAM)

In [ ]:
#@title Configuration

# --- Change these ---
CHARACTER = "debi-marlene"  #@param {type:"string"}
USE_FRAMEWORK = "llamafactory"  #@param ["llamafactory", "ms-swift"]
HF_MODEL_ID = "Qwen/Qwen2.5-Omni-7B"  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}
DRIVE_BACKUP_DIR = f"/content/drive/MyDrive/qwen25_omni_backup/{CHARACTER}"  #@param {type:"string"}

# --- Training hyperparameters ---
NUM_EPOCHS = 3  #@param {type:"integer"}
LEARNING_RATE = 1e-4  #@param {type:"number"}
LORA_RANK = 16  #@param {type:"integer"}
LORA_ALPHA = 32  #@param {type:"integer"}
BATCH_SIZE = 1  #@param {type:"integer"}
GRADIENT_ACCUMULATION = 8  #@param {type:"integer"}
MAX_LENGTH = 2048  #@param {type:"integer"}
SAVE_STEPS = 100  #@param {type:"integer"}

print(f"Character: {CHARACTER}")
print(f"Framework: {USE_FRAMEWORK}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

In [ ]:
#@title GPU Check
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
assert torch.cuda.is_available(), "GPU not available"
# Qwen2.5-Omni-7B with LoRA needs ~35GB VRAM
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
if vram_gb < 35:
    print(f"[WARNING] {vram_gb:.0f}GB VRAM detected. A100 40GB+ recommended.")
    print("Consider using QLoRA (4-bit) if you have less VRAM.")

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"Backup dir: {DRIVE_BACKUP_DIR}")

---
## 1. Data Preparation

### Dataset Format

Both LLaMA-Factory and ms-swift expect audio files + conversation pairs.

**Directory structure:**
```
data/
  audio/           # .wav or .mp3 files
    0001.wav
    0002.wav
    ...
  train.json       # conversation data
  dataset_info.json  # LLaMA-Factory only
```

**train.json format (ShareGPT style with audio):**
```json
[
  {
    "messages": [
      {"role": "system", "content": "..."},
      {"role": "user", "content": "<audio>"},
      {"role": "assistant", "content": "response text"}
    ],
    "audios": ["audio/0001.wav"]
  }
]
```

The `<audio>` tag in user content tells the model where to insert audio features.
The number of `<audio>` tags must match the number of paths in `audios`.

In [ ]:
#@title 1-1. Create example data (replace with your real data)
import json
import os
import numpy as np
import soundfile as sf

DATA_DIR = "/content/data"
AUDIO_DIR = f"{DATA_DIR}/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

# --- System prompt for character ---
# Debi & Marlene are twin characters from Eternal Return
SYSTEM_PROMPT = (
    "너는 이터널리턴의 캐릭터 데비&마를렌이야. "
    "데비는 밝고 활발한 성격이고, 마를렌은 차분하고 지적인 성격이야. "
    "둘은 항상 함께 행동하며, 대화할 때 두 캐릭터가 번갈아 말하기도 해. "
    "한국어로 답변하고, 캐릭터 말투를 유지해."
)

# --- Example data ---
# In real usage, you'd have:
#   - Actual recorded audio from Discord users asking questions
#   - Corresponding character responses written in Debi/Marlene's voice
examples = [
    {
        "audio_text": "오늘 날씨 어때?",  # what the audio contains (for reference only)
        "response": "데비: 오늘 날씨 완전 좋아! 밖에 나가서 놀자~ 마를렌: 자외선 지수가 높으니 선크림은 바르고 나가."
    },
    {
        "audio_text": "같이 게임할래?",
        "response": "데비: 게임?! 당연하지! 뭐 할 거야? 마를렌: 루미아 섬에서 만나자. 준비는 됐어."
    },
    {
        "audio_text": "요즘 뭐하고 지내?",
        "response": "데비: 나? 마를렌이랑 같이 새로운 실험하고 있어! 엄청 재밌어! 마를렌: ...위험한 실험은 아니니까 걱정하지 마."
    },
]

# Generate dummy audio files (replace with real audio in production)
# In practice, upload your Discord audio recordings to data/audio/
SAMPLE_RATE = 16000
for i, ex in enumerate(examples):
    audio_path = f"{AUDIO_DIR}/{i:04d}.wav"
    if not os.path.exists(audio_path):
        # 3 seconds of silence as placeholder
        dummy_audio = np.zeros(SAMPLE_RATE * 3, dtype=np.float32)
        sf.write(audio_path, dummy_audio, SAMPLE_RATE)

print(f"Created {len(examples)} example audio files in {AUDIO_DIR}")
print("[NOTE] Replace these with real audio recordings for actual training!")

In [ ]:
#@title 1-2. Build training JSON

train_data = []
for i, ex in enumerate(examples):
    entry = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "<audio>"},
            {"role": "assistant", "content": ex["response"]}
        ],
        "audios": [f"audio/{i:04d}.wav"]
    }
    train_data.append(entry)

# Save
train_path = f"{DATA_DIR}/train.json"
with open(train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(train_data)} examples to {train_path}")
print("\nSample entry:")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))

In [ ]:
#@title 1-3. Upload your real data (optional)
# If you have real data on Google Drive, copy it here
#
# Expected structure in Drive:
#   qwen25_omni_backup/{CHARACTER}/data/
#     audio/
#       0001.wav
#       0002.wav
#       ...
#     train.json
#
# Uncomment below to copy from Drive:

# import shutil
# drive_data = f"{DRIVE_BACKUP_DIR}/data"
# if os.path.exists(drive_data):
#     shutil.copytree(drive_data, DATA_DIR, dirs_exist_ok=True)
#     with open(f"{DATA_DIR}/train.json", "r") as f:
#         train_data = json.load(f)
#     print(f"Loaded {len(train_data)} examples from Drive")
# else:
#     print(f"No data found at {drive_data}, using example data")

In [ ]:
#@title 1-4. Create dataset_info.json (LLaMA-Factory only)

dataset_info = {
    f"{CHARACTER}_audio": {
        "file_name": "train.json",
        "formatting": "sharegpt",
        "columns": {
            "messages": "messages",
            "audios": "audios"
        },
        "tags": {
            "role_tag": "role",
            "content_tag": "content",
            "user_tag": "user",
            "assistant_tag": "assistant",
            "system_tag": "system"
        }
    }
}

info_path = f"{DATA_DIR}/dataset_info.json"
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"Saved dataset_info.json to {info_path}")
print(json.dumps(dataset_info, ensure_ascii=False, indent=2))

---
## 2A. LLaMA-Factory (Primary)

PR #7537 added Qwen2.5-Omni Thinker support.
- Template: `qwen2_omni`
- Only trains the Thinker (text LLM) part
- Audio/vision encoders are frozen by default

In [ ]:
#@title 2A-1. Install LLaMA-Factory
if USE_FRAMEWORK == "llamafactory":
    # LLaMA-Factory requires transformers >= 4.51.0 for Qwen2.5-Omni
    !pip install -q "transformers>=4.51.3"
    !pip install -q "llamafactory[torch,metrics]>=0.9.2"
    # qwen-omni-utils for audio/video processing
    !pip install -q qwen-omni-utils[decord]
    !pip install -q flash-attn --no-build-isolation
    print("\n--- Installation complete ---")
    !llamafactory-cli version
else:
    print("Skipping LLaMA-Factory install (using ms-swift)")

In [ ]:
#@title 2A-2. Write training YAML config
if USE_FRAMEWORK == "llamafactory":
    OUTPUT_DIR = f"/content/output/{CHARACTER}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    yaml_config = f"""### Qwen2.5-Omni-7B Thinker LoRA SFT
### Template: qwen2_omni (from PR #7537)

### Model
model_name_or_path: {HF_MODEL_ID}

### Training method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: {LORA_RANK}
lora_alpha: {LORA_ALPHA}
lora_dropout: 0.05

### Dataset
dataset: {CHARACTER}_audio
dataset_dir: {DATA_DIR}
template: qwen2_omni
cutoff_len: {MAX_LENGTH}
overwrite_cache: true
preprocessing_num_workers: 4

### Freeze encoders (only train LLM/Thinker)
freeze_vision_tower: true
freeze_multi_modal_projector: false

### Output
output_dir: {OUTPUT_DIR}
logging_steps: 5
save_steps: {SAVE_STEPS}
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true

### Training hyperparameters
per_device_train_batch_size: {BATCH_SIZE}
gradient_accumulation_steps: {GRADIENT_ACCUMULATION}
learning_rate: {LEARNING_RATE}
num_train_epochs: {NUM_EPOCHS}
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
gradient_checkpointing: true
ddp_timeout: 180000000

### Eval
val_size: 0.05
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: {SAVE_STEPS}
"""

    config_path = "/content/train_config.yaml"
    with open(config_path, "w") as f:
        f.write(yaml_config)

    print(f"Config saved to {config_path}")
    print(yaml_config)

In [ ]:
#@title 2A-3. Login to Hugging Face (for gated model access)
if USE_FRAMEWORK == "llamafactory" and HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}
    print("Logged in to HF")
elif USE_FRAMEWORK == "llamafactory":
    print("[WARNING] No HF_TOKEN set. If model is gated, login manually:")
    print("  !huggingface-cli login")

In [ ]:
#@title 2A-4. Start Training (LLaMA-Factory)
if USE_FRAMEWORK == "llamafactory":
    !llamafactory-cli train /content/train_config.yaml

In [ ]:
#@title 2A-5. Test inference (LLaMA-Factory)
if USE_FRAMEWORK == "llamafactory":
    # Create inference config
    infer_yaml = f"""### Inference config
model_name_or_path: {HF_MODEL_ID}
adapter_name_or_path: {OUTPUT_DIR}
template: qwen2_omni
finetuning_type: lora
"""
    with open("/content/infer_config.yaml", "w") as f:
        f.write(infer_yaml)

    # Interactive chat (text-only test)
    print("Starting interactive chat...")
    print("Type a message to test. Use 'exit' to quit.")
    !llamafactory-cli chat /content/infer_config.yaml

In [ ]:
#@title 2A-6. Programmatic inference with audio (LLaMA-Factory)
if USE_FRAMEWORK == "llamafactory":
    from llamafactory.chat import ChatModel

    args = {
        "model_name_or_path": HF_MODEL_ID,
        "adapter_name_or_path": OUTPUT_DIR,
        "template": "qwen2_omni",
        "finetuning_type": "lora",
    }
    chat_model = ChatModel(args)

    # Test with audio file
    test_audio = f"{AUDIO_DIR}/0000.wav"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "<audio>"},
    ]

    response = ""
    for token in chat_model.stream_chat(
        messages,
        audios=[test_audio],
    ):
        response += token

    print(f"Response: {response}")
    del chat_model

---
## 2B. ms-swift (Backup)

Official ModelScope framework, also supports Qwen2.5-Omni Thinker SFT.
- `freeze_vit=True` and `freeze_aligner=True` to only train LLM
- Supports padding-free training
- Built-in Qwen2.5-Omni template

In [ ]:
#@title 2B-1. Install ms-swift
if USE_FRAMEWORK == "ms-swift":
    !pip install -q "ms-swift[llm]>=3.5"
    !pip install -q qwen-omni-utils[decord]
    !pip install -q flash-attn --no-build-isolation
    !pip install -q "transformers>=4.51.3"
    print("\n--- Installation complete ---")
    !swift --version
else:
    print("Skipping ms-swift install (using LLaMA-Factory)")

In [ ]:
#@title 2B-2. Prepare ms-swift dataset format
if USE_FRAMEWORK == "ms-swift":
    # ms-swift uses a slightly different format
    # It supports OpenAI-style messages with <audio> tags
    # Audio paths are relative or absolute
    
    swift_data = []
    for i, ex in enumerate(examples):
        entry = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "<audio>"},
                {"role": "assistant", "content": ex["response"]}
            ],
            "audios": [f"{AUDIO_DIR}/{i:04d}.wav"]
        }
        swift_data.append(entry)

    swift_path = f"{DATA_DIR}/train_swift.jsonl"
    with open(swift_path, "w", encoding="utf-8") as f:
        for entry in swift_data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Saved {len(swift_data)} examples to {swift_path}")

In [ ]:
#@title 2B-3. Login to Hugging Face
if USE_FRAMEWORK == "ms-swift" and HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}
    print("Logged in to HF")

In [ ]:
#@title 2B-4. Start Training (ms-swift CLI)
if USE_FRAMEWORK == "ms-swift":
    OUTPUT_DIR = f"/content/output/{CHARACTER}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    !PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True' \
    swift sft \
        --model {HF_MODEL_ID} \
        --dataset {DATA_DIR}/train_swift.jsonl \
        --train_type lora \
        --torch_dtype bfloat16 \
        --attn_impl flash_attn \
        --num_train_epochs {NUM_EPOCHS} \
        --per_device_train_batch_size {BATCH_SIZE} \
        --per_device_eval_batch_size 1 \
        --learning_rate {LEARNING_RATE} \
        --lora_rank {LORA_RANK} \
        --lora_alpha {LORA_ALPHA} \
        --target_modules all-linear \
        --freeze_vit true \
        --freeze_aligner true \
        --gradient_accumulation_steps {GRADIENT_ACCUMULATION} \
        --gradient_checkpointing true \
        --eval_steps {SAVE_STEPS} \
        --save_steps {SAVE_STEPS} \
        --save_total_limit 3 \
        --logging_steps 5 \
        --max_length {MAX_LENGTH} \
        --output_dir {OUTPUT_DIR} \
        --warmup_ratio 0.1 \
        --split_dataset_ratio 0.05 \
        --dataloader_num_workers 4

In [ ]:
#@title 2B-5. Start Training (ms-swift Python API, alternative)
# Uncomment to use Python API instead of CLI

# if USE_FRAMEWORK == "ms-swift":
#     from swift.llm import sft_main, TrainArguments
#     import os
#
#     OUTPUT_DIR = f"/content/output/{CHARACTER}"
#     os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
#
#     sft_main(TrainArguments(
#         model=HF_MODEL_ID,
#         dataset=f"{DATA_DIR}/train_swift.jsonl",
#         train_type='lora',
#         torch_dtype='bfloat16',
#         attn_impl='flash_attn',
#         num_train_epochs=NUM_EPOCHS,
#         per_device_train_batch_size=BATCH_SIZE,
#         per_device_eval_batch_size=1,
#         learning_rate=LEARNING_RATE,
#         lora_rank=LORA_RANK,
#         lora_alpha=LORA_ALPHA,
#         target_modules='all-linear',
#         freeze_vit=True,
#         freeze_aligner=True,
#         gradient_accumulation_steps=GRADIENT_ACCUMULATION,
#         gradient_checkpointing=True,
#         eval_steps=SAVE_STEPS,
#         save_steps=SAVE_STEPS,
#         save_total_limit=3,
#         logging_steps=5,
#         max_length=MAX_LENGTH,
#         output_dir=OUTPUT_DIR,
#         warmup_ratio=0.1,
#         split_dataset_ratio=0.05,
#         dataloader_num_workers=4,
#     ))

In [ ]:
#@title 2B-6. Test inference (ms-swift)
if USE_FRAMEWORK == "ms-swift":
    # Find the latest checkpoint
    import glob
    checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
    if checkpoints:
        latest_ckpt = checkpoints[-1]
        print(f"Using checkpoint: {latest_ckpt}")

        !swift infer \
            --model {HF_MODEL_ID} \
            --adapters {latest_ckpt} \
            --torch_dtype bfloat16 \
            --attn_impl flash_attn \
            --max_new_tokens 512 \
            --stream true
    else:
        print("No checkpoints found")

In [ ]:
#@title 2B-7. Programmatic inference with audio (ms-swift)
if USE_FRAMEWORK == "ms-swift":
    from swift.llm import PtEngine, InferRequest, RequestConfig
    import glob

    checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
    latest_ckpt = checkpoints[-1] if checkpoints else None

    if latest_ckpt:
        engine = PtEngine(
            HF_MODEL_ID,
            adapters=[latest_ckpt],
            torch_dtype='bfloat16',
            attn_impl='flash_attn',
        )

        test_audio = f"{AUDIO_DIR}/0000.wav"
        infer_request = InferRequest(
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "<audio>"},
            ],
            audios=[test_audio],
        )
        request_config = RequestConfig(temperature=0.7, max_tokens=512)
        resp_list = engine.infer([infer_request], request_config)
        print(f"Response: {resp_list[0].choices[0].message.content}")
        del engine
    else:
        print("No checkpoints found")

---
## 3. Save & Export Adapter

In [ ]:
#@title 3-1. Backup adapter to Google Drive
import shutil
import glob

OUTPUT_DIR = f"/content/output/{CHARACTER}"

# Find the best/latest checkpoint
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
adapter_files = glob.glob(f"{OUTPUT_DIR}/adapter_*") + glob.glob(f"{OUTPUT_DIR}/lora_*")

# For LLaMA-Factory, the final adapter is in OUTPUT_DIR directly
# For ms-swift, it's in the checkpoint folder
source_dir = OUTPUT_DIR
if checkpoints and USE_FRAMEWORK == "ms-swift":
    source_dir = checkpoints[-1]

backup_dest = f"{DRIVE_BACKUP_DIR}/adapter"
if os.path.exists(backup_dest):
    shutil.rmtree(backup_dest)

# Copy only adapter files (not full model)
os.makedirs(backup_dest, exist_ok=True)
for f in os.listdir(source_dir):
    src = os.path.join(source_dir, f)
    if os.path.isfile(src) and not f.startswith("global_step"):
        shutil.copy2(src, backup_dest)

print(f"Adapter backed up to: {backup_dest}")
!ls -lh {backup_dest}/

In [ ]:
#@title 3-2. Upload adapter to Hugging Face Hub (optional)
UPLOAD_TO_HF = False  #@param {type:"boolean"}
HF_REPO_ID = f"2R4mi/qwen25-omni-{CHARACTER}"  #@param {type:"string"}

if UPLOAD_TO_HF and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()

    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=backup_dest,
        repo_id=HF_REPO_ID,
        commit_message=f"{CHARACTER} LoRA adapter - Qwen2.5-Omni-7B",
    )
    print(f"Uploaded to: https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skipping HF upload")

In [ ]:
#@title 3-3. Backup training loss plot
loss_plot = f"{OUTPUT_DIR}/training_loss.png"
if os.path.exists(loss_plot):
    shutil.copy2(loss_plot, DRIVE_BACKUP_DIR)
    from IPython.display import Image, display
    display(Image(filename=loss_plot))
else:
    # Try trainer_state.json for manual plotting
    state_file = f"{OUTPUT_DIR}/trainer_state.json"
    if os.path.exists(state_file):
        import matplotlib.pyplot as plt
        with open(state_file) as f:
            state = json.load(f)
        losses = [(x["step"], x["loss"]) for x in state["log_history"] if "loss" in x]
        if losses:
            steps, loss_vals = zip(*losses)
            plt.figure(figsize=(10, 5))
            plt.plot(steps, loss_vals)
            plt.xlabel("Step")
            plt.ylabel("Loss")
            plt.title(f"Training Loss - {CHARACTER}")
            plt.grid(True, alpha=0.3)
            plt.savefig(f"{DRIVE_BACKUP_DIR}/training_loss.png", dpi=150)
            plt.show()
    else:
        print("No loss data found")

---
## 4. Production Inference

After training, use the adapter in your Discord bot.

The flow:
1. User speaks in Discord voice channel
2. Bot captures audio -> saves as .wav
3. Qwen2.5-Omni + LoRA adapter processes audio -> character text
4. Text -> CosyVoice3 TTS -> audio response

In [ ]:
#@title 4-1. Standalone inference example (for production)

# This shows how to load the model + adapter for inference
# Adapt this for your Discord bot's TTS pipeline

INFERENCE_CODE = '''
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from peft import PeftModel

# --- Config ---
BASE_MODEL = "Qwen/Qwen2.5-Omni-7B"
ADAPTER_PATH = "path/to/adapter"  # local or HF repo
SYSTEM_PROMPT = (
    "너는 이터널리턴의 캐릭터 데비&마를렌이야. "
    "데비는 밝고 활발한 성격이고, 마를렌은 차분하고 지적인 성격이야. "
    "한국어로 답변하고, 캐릭터 말투를 유지해."
)

# --- Load model ---
processor = Qwen2_5OmniProcessor.from_pretrained(BASE_MODEL)
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

# --- Inference function ---
def generate_response(audio_path: str) -> str:
    """Process audio input and generate character response text."""
    from qwen_omni_utils import process_mm_info

    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "audio", "audio": audio_path}]},
    ]

    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(
        text=text, audio=audios, images=images, videos=videos,
        return_tensors="pt", padding=True,
    )
    inputs = inputs.to(model.device).to(model.dtype)

    with torch.no_grad():
        text_ids = model.generate(
            **inputs,
            use_audio_in_video=False,
            return_audio=False,  # We only need text, TTS is separate
            thinker_do_sample=True,
            thinker_temperature=0.7,
            thinker_max_new_tokens=512,
        )

    response = processor.batch_decode(
        text_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return response

# --- Usage ---
# text = generate_response("discord_audio.wav")
# Then send `text` to CosyVoice3 TTS
'''

print(INFERENCE_CODE)

# Save for reference
with open(f"{DRIVE_BACKUP_DIR}/inference_example.py", "w") as f:
    f.write(INFERENCE_CODE)
print(f"\nSaved to {DRIVE_BACKUP_DIR}/inference_example.py")

---
## 5. Data Preparation Script

Helper to convert your raw audio+text pairs into the training format.

In [ ]:
#@title 5-1. Build dataset from audio folder + CSV

BUILD_DATASET_CODE = '''
"""Build training dataset from audio files + text responses.

Expected input:
  responses.csv with columns: audio_file, response
  audio/ directory with .wav files

Example CSV:
  audio_file,response
  audio/0001.wav,"데비: 안녕! 마를렌: 반가워."
  audio/0002.wav,"데비: 같이 놀자! 마를렌: 좋아."
"""
import csv
import json
import os

SYSTEM_PROMPT = (
    "너는 이터널리턴의 캐릭터 데비&마를렌이야. "
    "데비는 밝고 활발한 성격이고, 마를렌은 차분하고 지적인 성격이야. "
    "한국어로 답변하고, 캐릭터 말투를 유지해."
)

def build_dataset(csv_path, output_path, audio_base_dir=""):
    data = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            audio_file = row["audio_file"]
            if audio_base_dir:
                audio_file = os.path.join(audio_base_dir, audio_file)

            entry = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": "<audio>"},
                    {"role": "assistant", "content": row["response"]},
                ],
                "audios": [audio_file],
            }
            data.append(entry)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Built {len(data)} examples -> {output_path}")
    return data

# Usage:
# build_dataset("responses.csv", "data/train.json")
'''

print(BUILD_DATASET_CODE)

In [ ]:
#@title 5-2. Multi-turn conversation format

# For multi-turn conversations with mixed audio + text:
multi_turn_example = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        # Turn 1: user speaks (audio)
        {"role": "user", "content": "<audio>"},
        {"role": "assistant", "content": "데비: 안녕! 오늘 기분 어때? 마를렌: 질문이 있으면 편하게 물어봐."},
        # Turn 2: user speaks again (another audio)
        {"role": "user", "content": "<audio>"},
        {"role": "assistant", "content": "데비: 이터널리턴?! 나 그거 진짜 잘해! 마를렌: 실험체 추천해줄까?"},
    ],
    # Two audio files for two user turns
    "audios": ["audio/turn1.wav", "audio/turn2.wav"]
}

print("Multi-turn example:")
print(json.dumps(multi_turn_example, ensure_ascii=False, indent=2))
print("\n[NOTE] Number of <audio> tags must match number of audio paths!")

---
## Troubleshooting

### Known Issues

1. **OOM (Out of Memory)**
   - Reduce `MAX_LENGTH` (e.g., 1024)
   - Reduce `BATCH_SIZE` to 1
   - Enable gradient checkpointing (already enabled)
   - Use QLoRA: set `quantization_bit: 4` in LLaMA-Factory YAML, or `--quant_method bnb --quant_bits 4` in ms-swift

2. **LLaMA-Factory audio bug (Issue #7552)**
   - Fixed in recent versions. Make sure `transformers>=4.51.3` and latest `llamafactory`
   - If audio features show all -1.5 (Issue #7633), update packages

3. **Performance degradation after fine-tuning (Issue #8146)**
   - Lower learning rate (1e-5 instead of 1e-4)
   - Fewer epochs (1-2)
   - More diverse training data
   - Larger LoRA rank does not always help

4. **Hanging with DeepSpeed ZeRO-3 (Issue #7767)**
   - Use single GPU without DeepSpeed on Colab
   - Or use ZeRO-2 if multi-GPU

5. **Qwen2.5-Omni only supports Thinker training**
   - Talker (audio generation) training is NOT supported yet
   - We only train text output, then use CosyVoice3 for TTS (which is better for our use case)